In [16]:
import pandas as pd
import numpy as np
import re

In [5]:
df = pd.read_parquet('commodities_close.parquet')
df.head()

Ticker,ZW=F,CC=F,HG=F,CT=F,CL=F,BZ=F,ZC=F,ZS=F,NG=F,SI=F,PL=F,RB=F,PA=F,GC=F,HO=F,SB=F,LE=F,HE=F,KC=F
Date,,,,,,,,,,,,,,,,,,,
1997-10-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,402.700012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997-10-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,405.299988,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,404.200012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997-11-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,406.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997-11-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,405.700012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
prices = df.loc['2002-01-01':]

In [8]:
prices.head()

Ticker,ZW=F,CC=F,HG=F,CT=F,CL=F,BZ=F,ZC=F,ZS=F,NG=F,SI=F,PL=F,RB=F,PA=F,GC=F,HO=F,SB=F,LE=F,HE=F,KC=F
Date,,,,,,,,,,,,,,,,,,,
2002-01-02,292.00,1296.0,0.6525,36.459999,21.010000,NaN,199.5,422.75,2.465,4.526,483.500000,0.6106,439.000000,278.899994,0.5786,7.71,70.519997,56.570000,47.349998
2002-01-03,292.25,1294.0,0.6580,36.139999,20.370001,NaN,202.0,418.50,2.268,4.599,485.500000,0.6045,435.200012,278.200012,0.5600,7.65,70.570000,55.599998,47.500000
2002-01-04,301.00,1386.0,0.6700,36.619999,21.620001,NaN,202.5,423.50,2.275,4.642,478.200012,0.6284,425.500000,278.899994,0.5859,7.64,70.070000,55.299999,48.549999
2002-01-07,308.00,1411.0,0.7095,37.560001,21.480000,NaN,203.5,428.00,2.272,4.660,476.299988,0.6234,424.000000,278.600006,0.5719,7.71,70.419998,55.299999,48.250000
2002-01-08,304.50,1389.0,0.7035,38.450001,21.250000,NaN,204.0,430.00,2.281,4.633,478.299988,0.6099,431.000000,278.899994,0.5682,7.80,70.120003,54.669998,48.549999


In [7]:
copper = prices['HG=F']
gold = prices['GC=F']
oil  = prices['CL=F']
wheat = prices['ZW=F']
corn = prices['ZS=F']
gas = prices['NG=F']
silver = prices['SI=F']

In [8]:
assets = {
    'copper': copper,
    'gold': gold,
    'oil': oil,
    'wheat': wheat,
    'corn': corn,
    'gas': gas,
    'silver': silver
}

def calc_rsi(series, period):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def generate_targets_features(assets, h_list=[1,5,10,20], k_list=[1,5,10,20,30], rsi_periods=[7,14,21]):
    all_data = {}
    max_k = max(k_list)
    max_h = max(h_list)
    
    for name, price in assets.items():
        df = pd.DataFrame(index=price.index)
        
        for h in h_list:
            df[f'target_{name}_h{h}'] = np.log(price.shift(-h) / price)
        
        for k in k_list:
            df[f'return_{name}_k{k}'] = np.log(price / price.shift(k))
            df[f'MA_{name}_{k}'] = price.shift(1).rolling(k).mean()
        
        for p in rsi_periods:
            df[f'RSI_{name}_{p}'] = calc_rsi(price.shift(1), p)
        
        df = df.iloc[max_k:-max_h] if max_h > 0 else df.iloc[max_k:]
        all_data[name] = df
    
    return all_data

features_targets = generate_targets_features(assets)

print(features_targets['gold'].head())

            target_gold_h1  target_gold_h5  target_gold_h10  target_gold_h20  \
Date                                                                           
2002-02-14       -0.004347       -0.021927        -0.005689        -0.033246   
2002-02-15       -0.017580       -0.019287        -0.005713        -0.021339   
2002-02-19       -0.003074        0.016238         0.002725        -0.001365   
2002-02-20        0.003074        0.013592         0.004097        -0.000684   
2002-02-21        0.000000        0.011867        -0.010629        -0.001365   

            return_gold_k1   MA_gold_1  return_gold_k5   MA_gold_5  \
Date                                                                 
2002-02-14        0.001002  299.399994        0.000000  300.680005   
2002-02-15       -0.004347  299.700012       -0.016947  300.680005   
2002-02-19       -0.017580  298.399994       -0.023261  299.660004   
2002-02-20       -0.003074  293.200012       -0.028333  298.280005   
2002-02-21        0

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered 

In [27]:
for name, df in features_targets.items():
    for col in df.columns:
        if col.startswith("target_"):
            out = df[[col] + [c for c in df.columns if not c.startswith("target_")]]
            out = out.replace([np.inf, -np.inf], np.nan).dropna()
            out.to_parquet(f"data/{name}_{col}.parquet")